# Session 1 — From CFD Export to Trustworthy Data

**Post-CFD Analysis with Python | Dr. Nuha Aljuneidi**

A CFD solver's output is a file, not an answer. Before any result is trustworthy, you must import it correctly, confirm the solver actually converged, check the data for defects, and show that the result does not depend on mesh resolution. This session builds that foundation.

## Learning outcomes
- Import a CFD-style export into a labeled pandas DataFrame, with units documented via column-naming convention (e.g. `pressure_Pa`), not tracked as a true unit type.
- Distinguish residual convergence from engineering (result) convergence.
- Run systematic data-quality checks before trusting any number.
- Perform a mesh-independence check and interpret the result.

## 0. The post-processing workflow

For every CFD result: **import → verify convergence → check data quality → check mesh independence → then, only then, analyze.**

Skipping straight to analysis is the single most common source of untrustworthy CFD-based engineering claims.

In [ ]:
# Run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
print("Environment ready.")

## 1. Importing a CFD export

Fluent and most solvers export point or surface data as CSV: spatial coordinates plus field variables. Below we synthesize a small export — 500 points sampled over a 2D domain — with pressure and velocity components, mimicking what `File > Export > Solution Data` would produce.

In [ ]:
n_points = 500
x_m = rng.uniform(0.0, 1.0, n_points)
y_m = rng.uniform(-0.2, 0.2, n_points)

U_inf_mps = 10.0
rho_kgpm3 = 1.225
p_inf_Pa = 101325.0

# Synthetic potential-flow-like field around a bump, plus noise
velocity_x_mps = U_inf_mps * (1.0 + 0.15 * np.exp(-((x_m - 0.5) ** 2) / 0.02)) + rng.normal(0, 0.05, n_points)
velocity_y_mps = 0.4 * np.sin(2 * np.pi * x_m) * np.exp(-y_m ** 2 / 0.02) + rng.normal(0, 0.03, n_points)
pressure_Pa = p_inf_Pa + 0.5 * rho_kgpm3 * (U_inf_mps ** 2 - (velocity_x_mps ** 2 + velocity_y_mps ** 2)) + rng.normal(0, 5, n_points)

df = pd.DataFrame({
    "x_m": x_m, "y_m": y_m,
    "velocity_x_mps": velocity_x_mps, "velocity_y_mps": velocity_y_mps,
    "pressure_Pa": pressure_Pa,
})
print(f"Imported {len(df)} points with columns: {list(df.columns)}")
df.head()

### Checkpoint 1 — Read the export like an engineer
Before using `df`, answer in a markdown cell or comments:
1. What are the units of every column, and are they SI?
2. What reference values (here, `U_inf_mps`, `p_inf_Pa`) will you need again later, and where should you store them so you don't retype them?
3. Is `pressure_Pa` gauge or absolute in this export? How would you tell from a real Fluent file?

In [ ]:
# TODO: store U_inf_mps, rho_kgpm3, and p_inf_Pa in a single dict "case_ref" you can reuse in later cells
case_ref = {}


## 2. Residual convergence vs. engineering convergence

A solver reports **residuals** — how well the governing equations are satisfied at each iteration. A small residual tells you the *math* converged. It does **not** tell you the *quantity you care about* (a force, a temperature, a pressure drop) has stopped changing. Always check both.

In [ ]:
n_iter = 400
iterations = np.arange(1, n_iter + 1)

# Synthetic residual histories: exponential decay + floor + noise
continuity_residual = 1e-1 * np.exp(-iterations / 60) + 5e-7 + rng.normal(0, 2e-8, n_iter).clip(min=0)
momentum_residual = 5e-2 * np.exp(-iterations / 55) + 3e-7 + rng.normal(0, 1e-8, n_iter).clip(min=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(iterations, continuity_residual, label="continuity")
ax.semilogy(iterations, momentum_residual, label="x-momentum")
ax.axhline(1e-6, color="k", linestyle="--", linewidth=1, label="1e-6 target")
ax.set_xlabel("Iteration")
ax.set_ylabel("Scaled residual")
ax.set_title("Residual convergence history")
ax.legend()
plt.tight_layout()
plt.show()

converged_iter = iterations[(continuity_residual < 1e-6) & (momentum_residual < 1e-6)]
print(f"Residuals first drop below 1e-6 at iteration {converged_iter.min() if len(converged_iter) else 'never'}.")

In [ ]:
# Now check the *engineering* quantity: a monitored drag-like force coefficient
force_history = 1.02 + 0.30 * np.exp(-iterations / 45) + rng.normal(0, 0.004, n_iter)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(iterations, force_history)
ax.set_xlabel("Iteration")
ax.set_ylabel("Monitored force coefficient")
ax.set_title("Engineering-quantity convergence")
plt.tight_layout()
plt.show()

last_50 = force_history[-50:]
force_drift = last_50.max() - last_50.min()
print(f"Force coefficient over last 50 iterations: mean={last_50.mean():.4f}, "
      f"drift={force_drift:.4f} ({100*force_drift/last_50.mean():.2f}% of mean) — "
      f"{'stable' if force_drift/last_50.mean() < 0.01 else 'NOT yet stable'}.")

### Checkpoint 2 — Two kinds of convergence
Residuals can flatten out near a numerical floor while a monitored force is still drifting — that is exactly what a stalled or under-relaxed case looks like. In your own words: why is it unsafe to report a drag coefficient the moment residuals cross 1e-6, without checking the force-history plot?

## 3. Data-quality checks

A converged, well-imported dataset can still have defects: missing values, duplicate points, out-of-range physical values, or a units mismatch. Run these checks on every import, before any analysis.

In [ ]:
def quality_report(frame, checks):
    """Run a dict of {name: boolean_series_or_bool} checks and print a pass/fail summary."""
    print(f"{'check':30s} {'result'}")
    print("-" * 45)
    all_pass = True
    for name, result in checks.items():
        passed = bool(result) if np.isscalar(result) else bool(np.all(result))
        all_pass &= passed
        print(f"{name:30s} {'PASS' if passed else 'FAIL'}")
    print("-" * 45)
    print("Overall:", "PASS" if all_pass else "FAIL — inspect before proceeding")
    return all_pass

checks = {
    "no missing values": df.notna().all().all(),
    "no duplicate points": ~df.duplicated(subset=["x_m", "y_m"]).any(),
    "pressure within physical range": df["pressure_Pa"].between(0.5 * p_inf_Pa, 1.5 * p_inf_Pa),
    "velocity magnitude below 3x U_inf": np.sqrt(df.velocity_x_mps**2 + df.velocity_y_mps**2) < 3 * U_inf_mps,
}
_ = quality_report(df, checks)

### Checkpoint 3 — Add a check
Add one more check to the `checks` dict: that every `x_m` value lies within the domain you intended to export (e.g., 0 to 1 m). Re-run the report. What would a FAIL here usually indicate about the export, versus about the simulation itself?

In [ ]:
# TODO: add an "x within domain bounds" check to a copy of the checks dict and re-run quality_report


## 4. Mesh-independence exercise

A result is not trustworthy until it stops changing as the mesh is refined. Below, three synthetic "meshes" of increasing resolution produce a monitored quantity (here, a probe pressure). We compare successive refinements the way you would with real mesh-convergence data.

The check below uses a simple percent-change heuristic — easy to compute, but not the rigorous engineering standard. The formal standard is a **Grid Convergence Index (GCI)** built from Richardson extrapolation, which accounts for the actual observed order of accuracy and refinement ratio instead of an arbitrary percentage. The Graduate/Advanced Extension below walks through that calculation; treat the percent-change check here as a fast first pass, not a substitute for it.

In [ ]:
cell_counts = np.array([50_000, 200_000, 800_000])
# Synthetic probe pressure approaching a "true" value as mesh refines (Richardson-style trend)
true_value = 250.0
probe_pressure_Pa = true_value + 40.0 / (cell_counts / cell_counts[0]) ** 1.2 + rng.normal(0, 0.5, 3)

for n, p in zip(cell_counts, probe_pressure_Pa):
    print(f"mesh = {n:>9,d} cells   probe pressure = {p:8.3f} Pa")

change_coarse_to_med = abs(probe_pressure_Pa[1] - probe_pressure_Pa[0])
change_med_to_fine = abs(probe_pressure_Pa[2] - probe_pressure_Pa[1])
refinement_ratio = change_med_to_fine / change_coarse_to_med if change_coarse_to_med else np.nan

print(f"\nChange (coarse->medium): {change_coarse_to_med:.3f} Pa")
print(f"Change (medium->fine):   {change_med_to_fine:.3f} Pa")
print(f"Ratio: {refinement_ratio:.2f} — result "
      f"{'passes the simple <5% heuristic (run the GCI calculation below to confirm rigorously)' if change_med_to_fine < 0.05 * abs(probe_pressure_Pa[2]) else 'fails even the simple heuristic; refine further'}.")

### Checkpoint 4 — Report the result correctly
Following the course's engineering standard, write one sentence reporting the fine-mesh probe pressure that includes: the value, the unit, the reference condition (which mesh), a numerical-quality statement (is it mesh-independent?), and one limitation (e.g., this is only a 3-point refinement study).

## Graduate/Advanced ExtensionReal mesh studies rarely have exactly the ideal refinement ratio $r=2$. Using the Richardson extrapolation formula $f_{exact} approx f_1 + rac{f_1 - f_2}{r^{p} - 1}$, estimate the extrapolated "exact" probe pressure from the medium and fine mesh values, assuming a nominal order of accuracy $p=2$ and refinement ratio $r = (800000/200000)^{1/2}$. This extrapolation is the core of the formal Grid Convergence Index (GCI) referenced in Section 4 — the GCI itself is this same quantity expressed as a fractional error with a safety factor applied. Compare it to the fine-mesh value and discuss whether the difference matters for your engineering decision.

## Exit ticket
In three sentences: state one thing you checked before trusting a CFD result today that you might have skipped before, one data-quality check you would add for your own project, and what mesh-independence evidence you would want to see before signing off on someone else's CFD result.

**Next:** Session 2 turns verified field data into publication-quality contour maps and profiles.